# Experiment 3.0 — Phase C hierarchical multi-$\tau_{\mathrm{syn}}$ SNN

## Scientific question

Phase B established that a two-layer feed-forward SNN can learn the gesture task and that
`timestep_ce` and `relative_10bin_ce` impose different temporal supervision. Experiment 3.0 asks:

> **Can a hierarchical heterogeneous multi-$\tau_{\mathrm{syn}}$ architecture capture both short local motion motifs and long gesture-scale temporal structure?**

### Architecture

\[
30
\rightarrow
\underbrace{128}_{s\in\{2,3\}}
\rightarrow
\underbrace{128}_{s\in\{2,3,4,5,6,7\}}
\rightarrow K
\]

- Hidden layer 1: 128 Synaptic-LIF neurons, shifts `[2, 3]`, allocated `[64, 64]`.
- Hidden layer 2: 128 Synaptic-LIF neurons, shifts `[2, 3, 4, 5, 6, 7]`, allocated as evenly as possible across 128 neurons: `[22, 21, 21, 21, 21, 22]`.
- The two extra neurons are placed at the fastest and slowest endpoints to avoid systematically favoring only fast or only slow groups.
- No recurrence.
- Hidden-layer $\tau_{\mathrm{syn}}$ values are **heterogeneous but fixed**: they are assigned before training and are not learnable.
- $\tau_{\mathrm{mem}}$, threshold, output-layer dynamics, optimizer, data split, and training protocol remain fixed.
- Linear hidden/output layers use `bias=False`.

### Objectives

Every training seed is run under both objectives:

1. `timestep_ce`
2. `relative_10bin_ce`

Five paired training seeds are used with one fixed user-disjoint split, giving:

\[
5\ \text{seeds}\times2\ \text{objectives}=10\ \text{full trainings}.
\]

The validation split selects the best epoch for each run. The test split is evaluated only after
that checkpoint has been selected.

### Diagnostics

In addition to classification metrics, this notebook records:

- validation balanced accuracy vs. epoch;
- hidden-layer and output firing rates;
- firing rate for each $\tau_{\mathrm{syn}}$ neuron group;
- spike raster plots with timestep on the x-axis and individual neurons on the y-axis.

In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import math
import os
import random
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

import snntorch as snn
from snntorch import surrogate
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "snn").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the writingRing repository root")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from snn.accel_reconstruction_eval.datasets import load_acceleration_data

print("Repository root:", REPO_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 1. Configuration

In [ ]:
DATASET_ROOTS = [
    REPO_ROOT / "outputs/action0_wavelets_0e5_1_2_4_8_sr_64",
    REPO_ROOT / "outputs/action1_wavelets_0e5_1_2_4_8_sr_64",
]

INCLUDED_LABELS = (
    "A", "B", "C", "D", "E", "X",
    "G", "H", "I", "J", "K", "L",
)

SPLIT_SEED = 12345
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15

SEEDS = (11, 23, 101, 40, 231)

OBJECTIVES = (
    "timestep_ce",
    "relative_10bin_ce",
)
N_RELATIVE_BINS = 10

HIDDEN_WIDTH = 128

LAYER1_SHIFTS = (2, 3)
LAYER1_COUNTS = (64, 64)

LAYER2_SHIFTS = (2, 3, 4, 5, 6, 7)
LAYER2_COUNTS = (22, 21, 21, 21, 21, 22)

if sum(LAYER1_COUNTS) != HIDDEN_WIDTH:
    raise ValueError("Layer-1 group counts must sum to HIDDEN_WIDTH")
if sum(LAYER2_COUNTS) != HIDDEN_WIDTH:
    raise ValueError("Layer-2 group counts must sum to HIDDEN_WIDTH")
if len(LAYER1_SHIFTS) != len(LAYER1_COUNTS):
    raise ValueError("Layer-1 shifts/counts length mismatch")
if len(LAYER2_SHIFTS) != len(LAYER2_COUNTS):
    raise ValueError("Layer-2 shifts/counts length mismatch")

# Keep Phase-B membrane, threshold and output-layer dynamics fixed.
TAU_MEM_MS = 22.54
TAU_SYN_OUT_MS = 77.47
THRESHOLD = 0.5
SURROGATE_SLOPE = 25.0
RESET_MECHANISM = "subtract"

SPIKE_REGULARIZATION = 0.0
FIRING_ONSET_THRESHOLD = 1e-3
ZERO_INPUT_DIAGNOSTIC_STEPS = 200

BATCH_SIZE = 128
NUM_WORKERS = 0
NUM_EPOCHS = 150
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = None

RESUME_EXISTING = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EXPERIMENT_ID = "experiment_3_0_phase_c_multitau_objectives"
RESULTS_DIR = REPO_ROOT / "notebooks/artifacts" / EXPERIMENT_ID
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
HISTORY_DIR = RESULTS_DIR / "histories"

print("Experiment:", EXPERIMENT_ID)
print("Objectives:", OBJECTIVES)
print("Seeds:", SEEDS)
print("Fixed split seed:", SPLIT_SEED)
print("Hidden width:", HIDDEN_WIDTH)
print("Layer 1:", list(zip(LAYER1_SHIFTS, LAYER1_COUNTS)))
print("Layer 2:", list(zip(LAYER2_SHIFTS, LAYER2_COUNTS)))
print("Full trainings:", len(OBJECTIVES) * len(SEEDS))
print("Epoch budget:", NUM_EPOCHS)
print("Device:", DEVICE)
print("Resume existing:", RESUME_EXISTING)

## 2. Reproducibility and shift helpers

In [ ]:
def derive_seed(master_seed: int, *parts: object) -> int:
    text = "|".join([str(master_seed), *(str(p) for p in parts)])
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "little", signed=False)


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)


def worker_init_fn(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def shift_to_alpha(shift: int) -> float:
    return float(1.0 - 2.0 ** (-int(shift)))


def alpha_to_tau_ms(alpha: float, sampling_rate_hz: float) -> float:
    dt_ms = 1000.0 / float(sampling_rate_hz)
    return float(-dt_ms / math.log(float(alpha)))


def build_alpha_vector(shifts, counts):
    return [
        shift_to_alpha(shift)
        for shift, count in zip(shifts, counts)
        for _ in range(int(count))
    ]


def build_group_slices(shifts, counts):
    groups = []
    start = 0
    for shift, count in zip(shifts, counts):
        stop = start + int(count)
        groups.append((int(shift), start, stop))
        start = stop
    return groups


LAYER1_ALPHA_VECTOR = build_alpha_vector(LAYER1_SHIFTS, LAYER1_COUNTS)
LAYER2_ALPHA_VECTOR = build_alpha_vector(LAYER2_SHIFTS, LAYER2_COUNTS)
LAYER1_GROUP_SLICES = build_group_slices(LAYER1_SHIFTS, LAYER1_COUNTS)
LAYER2_GROUP_SLICES = build_group_slices(LAYER2_SHIFTS, LAYER2_COUNTS)

assert len(LAYER1_ALPHA_VECTOR) == HIDDEN_WIDTH
assert len(LAYER2_ALPHA_VECTOR) == HIDDEN_WIDTH

## 3. Load and validate unsigned event data

In [ ]:
data = load_acceleration_data(
    DATASET_ROOTS,
    repository_root=REPO_ROOT,
    require_reconstruction=False,
)

sampling_rates = {float(m.sampling_rate_hz) for m in data.producer_metadatas}
if len(sampling_rates) != 1:
    raise ValueError(f"Expected one shared sampling rate, got {sampling_rates}")
SAMPLING_RATE_HZ = sampling_rates.pop()

event_contracts = set()
for metadata in data.producer_metadatas:
    raw = metadata.raw
    event_representation = raw.get("event_representation")
    event_feature_schema = raw.get("event_feature_schema")
    event_channel_count = raw.get("event_channel_count")
    encoder_spec_sha256 = raw.get("spike_encoder_spec_sha256")

    if event_representation != "unsigned":
        raise ValueError(
            "Requires event_representation='unsigned'; "
            f"got {event_representation!r}"
        )
    if not isinstance(event_feature_schema, str) or not event_feature_schema:
        raise ValueError("Missing event_feature_schema")
    if not isinstance(event_channel_count, int) or event_channel_count <= 0:
        raise ValueError("event_channel_count must be positive")
    if event_channel_count != metadata.channel_count - 6:
        raise ValueError(
            "Expected six auxiliary IMU channels after event channels; "
            f"event={event_channel_count}, total={metadata.channel_count}"
        )
    if not isinstance(encoder_spec_sha256, str) or len(encoder_spec_sha256) != 64:
        raise ValueError("Missing spike_encoder_spec_sha256")

    event_contracts.add(
        (
            event_representation,
            event_feature_schema,
            event_channel_count,
            encoder_spec_sha256,
        )
    )

if len(event_contracts) != 1:
    raise ValueError(f"Incompatible event contracts: {event_contracts}")

(
    EVENT_REPRESENTATION,
    EVENT_FEATURE_SCHEMA,
    INPUT_CHANNELS,
    ENCODER_SPEC_SHA256,
) = event_contracts.pop()

if INPUT_CHANNELS != 30:
    raise ValueError(
        f"Experiment 3.0 expects 30 polarity-split event channels, got {INPUT_CHANNELS}"
    )

rows = []
included = None if INCLUDED_LABELS is None else set(map(str, INCLUDED_LABELS))
padded_lengths = set()

for package_index, package in enumerate(data.packages):
    padded_lengths.add(int(package.padded_spike_imu.shape[1]))

    for segment_index, label in enumerate(package.labels.astype(str)):
        label = str(label)
        if included is not None and label not in included:
            continue

        valid_length = int(package.valid_lengths[segment_index])
        if valid_length <= 0:
            raise ValueError("valid_length must be positive")

        event_values = np.asarray(
            package.padded_spike_imu[
                segment_index, :valid_length, :INPUT_CHANNELS
            ]
        )
        if np.any(~np.isfinite(event_values)):
            raise ValueError("Non-finite event values")
        if np.any(event_values < 0.0):
            raise ValueError(
                "Unsigned event contract violated for "
                f"{package.user}/action_{package.action}/{segment_index}"
            )

        rows.append(
            {
                "package_index": package_index,
                "segment_index": segment_index,
                "user": str(package.user),
                "action": str(package.action),
                "label": label,
                "valid_length": valid_length,
                "sample_id": f"{package.user}/action_{package.action}/{segment_index}",
            }
        )

if len(padded_lengths) != 1:
    raise ValueError(f"Expected one padded length, got {padded_lengths}")
PADDED_LENGTH = padded_lengths.pop()

manifest = pd.DataFrame(rows)
if manifest.empty:
    raise ValueError("No samples remain after label filtering")

labels_sorted = sorted(manifest["label"].unique().tolist())
CLASS_TO_IDX = {label: i for i, label in enumerate(labels_sorted)}
IDX_TO_CLASS = {i: label for label, i in CLASS_TO_IDX.items()}
manifest["label_idx"] = manifest["label"].map(CLASS_TO_IDX).astype(int)
NUM_CLASSES = len(CLASS_TO_IDX)

DT_MS = 1000.0 / SAMPLING_RATE_HZ
BETA = float(math.exp(-DT_MS / TAU_MEM_MS))
OUTPUT_ALPHA = float(math.exp(-DT_MS / TAU_SYN_OUT_MS))

tau_rows = []
for layer_name, shifts, counts in (
    ("L1", LAYER1_SHIFTS, LAYER1_COUNTS),
    ("L2", LAYER2_SHIFTS, LAYER2_COUNTS),
):
    for shift, count in zip(shifts, counts):
        alpha = shift_to_alpha(shift)
        tau_rows.append(
            {
                "layer": layer_name,
                "shift": shift,
                "neurons": count,
                "alpha": alpha,
                "nominal_tau_syn_ms": alpha_to_tau_ms(alpha, SAMPLING_RATE_HZ),
            }
        )

tau_table = pd.DataFrame(tau_rows)

print(
    f"samples={len(manifest)}, classes={NUM_CLASSES}, "
    f"input_channels={INPUT_CHANNELS}"
)
print(f"sampling_rate={SAMPLING_RATE_HZ} Hz, padded_length={PADDED_LENGTH}")
print(
    f"beta={BETA:.6f}, output_alpha={OUTPUT_ALPHA:.6f}, "
    f"threshold={THRESHOLD}"
)
display(tau_table)

## 4. One fixed user-disjoint split

In [ ]:
def make_user_split(manifest: pd.DataFrame) -> pd.DataFrame:
    users = sorted(manifest["user"].unique().tolist())
    rng = np.random.default_rng(SPLIT_SEED)
    perm = np.array(users, dtype=object)
    rng.shuffle(perm)

    n = len(perm)
    n_train = max(1, int(np.floor(TRAIN_FRACTION * n)))
    n_val = max(1, int(np.floor(VAL_FRACTION * n)))
    if n_train + n_val >= n:
        n_train, n_val = n - 2, 1

    train_users = set(perm[:n_train].tolist())
    val_users = set(perm[n_train:n_train + n_val].tolist())

    out = manifest.copy()
    out["split"] = out["user"].map(
        lambda u: "train" if u in train_users else (
            "val" if u in val_users else "test"
        )
    )
    return out


FIXED_SPLIT_MANIFEST = make_user_split(manifest)

split_summary = FIXED_SPLIT_MANIFEST.groupby("split").agg(
    users=("user", "nunique"),
    samples=("label", "size"),
    classes_present=("label", "nunique"),
)
display(split_summary)

for split_name, sdf in FIXED_SPLIT_MANIFEST.groupby("split"):
    if sdf["label"].nunique() != NUM_CLASSES:
        print(
            f"WARNING: split={split_name!r} contains "
            f"{sdf['label'].nunique()}/{NUM_CLASSES} classes"
        )

## 5. Dataset and paired DataLoaders

In [ ]:
class EventSNNDataset(Dataset):
    def __init__(self, data, subset_manifest: pd.DataFrame):
        self.data = data
        self.df = subset_manifest.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        package = self.data.packages[int(row.package_index)]
        segment_index = int(row.segment_index)
        valid_length = int(row.valid_length)

        x = np.asarray(
            package.padded_spike_imu[
                segment_index, :, :INPUT_CHANNELS
            ],
            dtype=np.float32,
        ).copy()

        if x.shape != (PADDED_LENGTH, INPUT_CHANNELS):
            raise ValueError(f"Unexpected input shape {x.shape}")
        if np.any(x[:valid_length] < 0.0):
            raise ValueError("Unsigned event contract violated")

        x[valid_length:] = 0.0
        valid_mask = np.arange(PADDED_LENGTH) < valid_length

        return {
            "x": torch.from_numpy(x),
            "label": torch.tensor(int(row.label_idx), dtype=torch.long),
            "valid_mask": torch.from_numpy(valid_mask),
            "valid_length": torch.tensor(valid_length, dtype=torch.long),
            "sample_id": str(row.sample_id),
        }


def make_loader(
    subset_manifest: pd.DataFrame,
    *,
    batch_size: int,
    shuffle: bool,
    seed: int,
):
    dataset = EventSNNDataset(data, subset_manifest)
    generator = torch.Generator().manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        generator=generator,
        worker_init_fn=worker_init_fn if NUM_WORKERS > 0 else None,
        pin_memory=torch.cuda.is_available(),
    )


def make_split_loaders(master_seed: int, include_test: bool = False):
    specs = [
        ("train", "train", True),
        ("train_eval", "train", False),
        ("val", "val", False),
    ]
    if include_test:
        specs.append(("test", "test", False))

    loaders = {}
    for name, split_name, shuffle in specs:
        subset = FIXED_SPLIT_MANIFEST[
            FIXED_SPLIT_MANIFEST.split == split_name
        ]
        loaders[name] = make_loader(
            subset,
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            # Deliberately objective-independent for paired runs.
            seed=derive_seed(master_seed, name, "loader"),
        )
    return loaders

## 6. Relative 10-bin aggregation

In [ ]:
def relative_temporal_bin_counts(
    output_spikes: torch.Tensor,
    valid_lengths: torch.Tensor,
    n_bins: int,
) -> torch.Tensor:
    """Return [B, n_bins, K] output-spike counts over relative gesture progress."""
    _, time_steps, _ = output_spikes.shape

    lengths = valid_lengths.to(
        output_spikes.device, dtype=torch.long
    ).clamp_min(1)

    t = torch.arange(
        time_steps,
        device=output_spikes.device,
    )[None, :]

    valid = t < lengths[:, None]

    bin_index = torch.div(
        t * n_bins,
        lengths[:, None],
        rounding_mode="floor",
    ).clamp_max(n_bins - 1)

    one_hot = F.one_hot(
        bin_index,
        num_classes=n_bins,
    ).to(output_spikes.dtype)

    one_hot = one_hot * valid.unsqueeze(-1).to(output_spikes.dtype)

    return torch.einsum(
        "btn,btk->bnk",
        one_hot,
        output_spikes,
    )

## 7. Two-layer hierarchical multi-$\tau_{\mathrm{syn}}$ SNN

The hidden-layer decay factors follow

\[
\alpha_s = 1-2^{-s}.
\]

At 64 Hz this gives approximately:

| shift | alpha | nominal tau_syn |
|---:|---:|---:|
| 2 | 0.75 | 54 ms |
| 3 | 0.875 | 117 ms |
| 4 | 0.9375 | 242 ms |
| 5 | 0.96875 | 492 ms |
| 6 | 0.984375 | 0.99 s |
| 7 | 0.9921875 | 1.99 s |

The hidden $\alpha$ vectors are assigned neuron-by-neuron and are not learnable.

In [ ]:
class PhaseCMultiTauSNN(nn.Module):
    """Two-layer feed-forward heterogeneous multi-tau_syn Synaptic-LIF network."""

    def __init__(
        self,
        *,
        objective: str,
        num_classes: int,
        input_channels: int,
        master_seed: int,
    ):
        super().__init__()

        if objective not in OBJECTIVES:
            raise ValueError(f"Unknown objective: {objective}")

        self.objective = str(objective)
        self.width = int(HIDDEN_WIDTH)
        self.depth = 2
        self.num_classes = int(num_classes)
        self.input_channels = int(input_channels)

        self.layer_group_specs = (
            tuple(zip(LAYER1_SHIFTS, LAYER1_COUNTS)),
            tuple(zip(LAYER2_SHIFTS, LAYER2_COUNTS)),
        )
        self.layer_group_slices = (
            tuple(LAYER1_GROUP_SLICES),
            tuple(LAYER2_GROUP_SLICES),
        )

        spike_grad = surrogate.fast_sigmoid(slope=SURROGATE_SLOPE)

        # Backbone weights are paired across the two objectives for each seed.
        seed_everything(derive_seed(master_seed, "shared_backbone_init"))

        self.fc1 = nn.Linear(input_channels, HIDDEN_WIDTH, bias=False)
        self.lif1 = snn.Synaptic(
            alpha=LAYER1_ALPHA_VECTOR,
            beta=BETA,
            threshold=THRESHOLD,
            spike_grad=spike_grad,
            learn_alpha=False,
            learn_beta=False,
            learn_threshold=False,
            reset_mechanism=RESET_MECHANISM,
        )

        self.fc2 = nn.Linear(HIDDEN_WIDTH, HIDDEN_WIDTH, bias=False)
        self.lif2 = snn.Synaptic(
            alpha=LAYER2_ALPHA_VECTOR,
            beta=BETA,
            threshold=THRESHOLD,
            spike_grad=spike_grad,
            learn_alpha=False,
            learn_beta=False,
            learn_threshold=False,
            reset_mechanism=RESET_MECHANISM,
        )

        self.fc_out = nn.Linear(HIDDEN_WIDTH, num_classes, bias=False)
        self.lif_out = snn.Synaptic(
            alpha=OUTPUT_ALPHA,
            beta=BETA,
            threshold=THRESHOLD,
            spike_grad=spike_grad,
            learn_alpha=False,
            learn_beta=False,
            learn_threshold=False,
            reset_mechanism=RESET_MECHANISM,
        )

        if self.objective == "relative_10bin_ce":
            seed_everything(
                derive_seed(master_seed, self.objective, "head_init")
            )
            self.readout_head = nn.Linear(
                N_RELATIVE_BINS * num_classes,
                num_classes,
                bias=True,
            )
        else:
            self.readout_head = None

    @property
    def backbone_parameter_count(self) -> int:
        modules = [self.fc1, self.fc2, self.fc_out]
        return sum(
            p.numel()
            for module in modules
            for p in module.parameters()
        )

    @property
    def readout_parameter_count(self) -> int:
        if self.readout_head is None:
            return 0
        return sum(p.numel() for p in self.readout_head.parameters())

    @property
    def total_parameter_count(self) -> int:
        return sum(p.numel() for p in self.parameters())

    @property
    def spike_neuron_count(self) -> int:
        return 2 * HIDDEN_WIDTH + self.num_classes

    def forward(self, x, valid_mask):
        self.lif1.reset_mem()
        self.lif2.reset_mem()
        self.lif_out.reset_mem()

        spk1_rec = []
        spk2_rec = []
        out_rec = []

        for step in range(x.shape[1]):
            h1, _, _ = self.lif1(self.fc1(x[:, step, :]))
            h2, _, _ = self.lif2(self.fc2(h1))
            spk_out, _, _ = self.lif_out(self.fc_out(h2))

            spk1_rec.append(h1)
            spk2_rec.append(h2)
            out_rec.append(spk_out)

        hidden_spikes = [
            torch.stack(spk1_rec, dim=1),
            torch.stack(spk2_rec, dim=1),
        ]
        output_spikes = torch.stack(out_rec, dim=1)

        mask_f = valid_mask.unsqueeze(-1).to(output_spikes.dtype)

        total_valid_spikes = (
            sum((spikes * mask_f).sum() for spikes in hidden_spikes)
            + (output_spikes * mask_f).sum()
        )

        return {
            "hidden_spikes": hidden_spikes,
            "output_spikes": output_spikes,
            "total_valid_spikes": total_valid_spikes,
        }

    def native_logits(
        self,
        out,
        valid_lengths,
        valid_mask=None,
    ):
        if self.objective == "timestep_ce":
            if valid_mask is None:
                t = torch.arange(
                    out["output_spikes"].shape[1],
                    device=out["output_spikes"].device,
                )[None, :]
                valid_mask = t < valid_lengths[:, None]

            return (
                out["output_spikes"]
                * valid_mask.unsqueeze(-1).to(out["output_spikes"].dtype)
            ).sum(dim=1)

        bins = relative_temporal_bin_counts(
            out["output_spikes"],
            valid_lengths,
            N_RELATIVE_BINS,
        )
        return self.readout_head(bins.flatten(start_dim=1))

    def training_loss(
        self,
        out,
        labels,
        valid_mask,
        valid_lengths,
    ):
        if self.objective == "timestep_ce":
            batch_size, time_steps, class_count = out["output_spikes"].shape
            targets = labels[:, None].expand(batch_size, time_steps)

            loss_per_step = F.cross_entropy(
                out["output_spikes"].reshape(
                    batch_size * time_steps,
                    class_count,
                ),
                targets.reshape(batch_size * time_steps),
                reduction="none",
            ).reshape(batch_size, time_steps)

            mask_f = valid_mask.to(loss_per_step.dtype)

            loss = (
                (loss_per_step * mask_f).sum()
                / mask_f.sum().clamp_min(1)
            )
        else:
            logits = self.native_logits(out, valid_lengths)
            loss = F.cross_entropy(logits, labels)

        if SPIKE_REGULARIZATION > 0:
            valid_neuron_time = (
                valid_mask.to(out["output_spikes"].dtype).sum().clamp_min(1)
                * self.spike_neuron_count
            )
            loss = (
                loss
                + SPIKE_REGULARIZATION
                * out["total_valid_spikes"]
                / valid_neuron_time
            )

        return loss

## 8. Metrics, per-$\tau$ firing rates, and zero-input diagnostics

In [ ]:
def classification_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
    }


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()

    total_loss = 0.0
    sample_count = 0
    valid_steps = 0
    y_true, y_pred = [], []

    hidden_spike_totals = [0.0, 0.0]
    group_spike_totals = {
        (layer_idx, shift): 0.0
        for layer_idx, groups in enumerate(model.layer_group_slices)
        for shift, _, _ in groups
    }

    output_spikes_total = 0.0
    silent_output_samples = 0

    for batch in loader:
        x = batch["x"].to(DEVICE, non_blocking=True)
        y = batch["label"].to(DEVICE, non_blocking=True)
        mask = batch["valid_mask"].to(DEVICE, non_blocking=True)
        lengths = batch["valid_length"].to(DEVICE, non_blocking=True)

        out = model(x, mask)
        loss = model.training_loss(out, y, mask, lengths)
        logits = model.native_logits(out, lengths, mask)
        pred = logits.argmax(dim=1)

        batch_size = len(y)
        batch_valid_steps = int(mask.sum().item())

        total_loss += float(loss.item()) * batch_size
        sample_count += batch_size
        valid_steps += batch_valid_steps

        y_true.extend(y.cpu().tolist())
        y_pred.extend(pred.cpu().tolist())

        mask_f = mask.unsqueeze(-1).to(out["output_spikes"].dtype)

        for layer_idx, spikes in enumerate(out["hidden_spikes"]):
            hidden_spike_totals[layer_idx] += float(
                (spikes * mask_f).sum().item()
            )

            for shift, start, stop in model.layer_group_slices[layer_idx]:
                group_spike_totals[(layer_idx, shift)] += float(
                    (
                        spikes[:, :, start:stop] * mask_f
                    ).sum().item()
                )

        batch_out = (out["output_spikes"] * mask_f).sum(dim=(1, 2))
        output_spikes_total += float(batch_out.sum().item())
        silent_output_samples += int((batch_out == 0).sum().item())

    metrics = classification_metrics(y_true, y_pred)
    metrics["loss"] = total_loss / max(sample_count, 1)

    total_hidden_spikes = sum(hidden_spike_totals)

    metrics["mean_hidden_firing_rate"] = (
        total_hidden_spikes
        / max(valid_steps * 2 * HIDDEN_WIDTH, 1)
    )

    for layer_idx, total in enumerate(hidden_spike_totals, start=1):
        metrics[f"hidden_{layer_idx}_firing_rate"] = (
            total / max(valid_steps * HIDDEN_WIDTH, 1)
        )

    for layer_idx, groups in enumerate(model.layer_group_slices):
        for shift, start, stop in groups:
            group_size = stop - start
            metrics[
                f"hidden_{layer_idx + 1}_shift_{shift}_firing_rate"
            ] = (
                group_spike_totals[(layer_idx, shift)]
                / max(valid_steps * group_size, 1)
            )

    metrics["output_firing_rate"] = (
        output_spikes_total
        / max(valid_steps * NUM_CLASSES, 1)
    )

    metrics["network_firing_rate"] = (
        (total_hidden_spikes + output_spikes_total)
        / max(valid_steps * model.spike_neuron_count, 1)
    )

    metrics["silent_output_fraction"] = (
        silent_output_samples / max(sample_count, 1)
    )
    metrics["samples"] = sample_count

    return metrics


@torch.no_grad()
def zero_input_activity(
    model,
    steps=ZERO_INPUT_DIAGNOSTIC_STEPS,
):
    model.eval()

    x = torch.zeros(
        1,
        steps,
        INPUT_CHANNELS,
        device=DEVICE,
    )
    mask = torch.ones(
        1,
        steps,
        dtype=torch.bool,
        device=DEVICE,
    )

    out = model(x, mask)

    result = {
        "zero_hidden_1_firing_rate": float(
            out["hidden_spikes"][0].mean().item()
        ),
        "zero_hidden_2_firing_rate": float(
            out["hidden_spikes"][1].mean().item()
        ),
        "zero_output_firing_rate": float(
            out["output_spikes"].mean().item()
        ),
    }

    result["zero_mean_hidden_firing_rate"] = float(
        np.mean(
            [
                result["zero_hidden_1_firing_rate"],
                result["zero_hidden_2_firing_rate"],
            ]
        )
    )
    return result

## 9. Fixed-budget training and best-validation-BA checkpoint

In [ ]:
def _cpu_state_dict(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def train_one_configuration(
    *,
    objective: str,
    master_seed: int,
    train_loader,
    train_eval_loader,
    val_loader,
):
    model = PhaseCMultiTauSNN(
        objective=objective,
        num_classes=NUM_CLASSES,
        input_channels=INPUT_CHANNELS,
        master_seed=master_seed,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_val_ba = -np.inf
    best_val_loss = np.inf
    best_epoch = -1
    firing_onset_epoch = None
    history_rows = []

    zero_before = zero_input_activity(model)

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()

        for batch in train_loader:
            x = batch["x"].to(DEVICE, non_blocking=True)
            y = batch["label"].to(DEVICE, non_blocking=True)
            mask = batch["valid_mask"].to(DEVICE, non_blocking=True)
            lengths = batch["valid_length"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            out = model(x, mask)
            loss = model.training_loss(
                out,
                y,
                mask,
                lengths,
            )

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    "Non-finite loss for "
                    f"objective={objective}, seed={master_seed}"
                )

            loss.backward()

            if GRAD_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRAD_CLIP_NORM,
                )

            optimizer.step()

        train_metrics = evaluate_model(model, train_eval_loader)
        val_metrics = evaluate_model(model, val_loader)

        if (
            firing_onset_epoch is None
            and val_metrics["output_firing_rate"] > FIRING_ONSET_THRESHOLD
        ):
            firing_onset_epoch = epoch

        val_ba = val_metrics["balanced_accuracy"]
        val_loss = val_metrics["loss"]

        improved = (
            val_ba > best_val_ba + 1e-12
            or (
                abs(val_ba - best_val_ba) <= 1e-12
                and val_loss < best_val_loss - 1e-12
            )
        )

        if improved:
            best_val_ba = val_ba
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = _cpu_state_dict(model)

        row = {
            "objective": objective,
            "seed": master_seed,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_balanced_accuracy": train_metrics["balanced_accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_hidden_1_firing_rate": train_metrics[
                "hidden_1_firing_rate"
            ],
            "train_hidden_2_firing_rate": train_metrics[
                "hidden_2_firing_rate"
            ],
            "train_output_firing_rate": train_metrics[
                "output_firing_rate"
            ],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_hidden_1_firing_rate": val_metrics[
                "hidden_1_firing_rate"
            ],
            "val_hidden_2_firing_rate": val_metrics[
                "hidden_2_firing_rate"
            ],
            "val_output_firing_rate": val_metrics[
                "output_firing_rate"
            ],
            "val_silent_output_fraction": val_metrics[
                "silent_output_fraction"
            ],
        }

        for key, value in val_metrics.items():
            if key.startswith("hidden_") and "_shift_" in key:
                row[f"val_{key}"] = value

        history_rows.append(row)

        if epoch == 1 or epoch % 10 == 0 or epoch == NUM_EPOCHS:
            print(
                f"epoch {epoch:>3d} | "
                f"train loss={train_metrics['loss']:.4f} "
                f"train BA={train_metrics['balanced_accuracy']:.4f} | "
                f"val loss={val_metrics['loss']:.4f} "
                f"val BA={val_metrics['balanced_accuracy']:.4f} | "
                f"L1 FR={val_metrics['hidden_1_firing_rate']:.4f} "
                f"L2 FR={val_metrics['hidden_2_firing_rate']:.4f} "
                f"out FR={val_metrics['output_firing_rate']:.4f}"
            )

    if best_state is None:
        raise RuntimeError("No best checkpoint selected")

    model.load_state_dict(best_state)
    zero_after = zero_input_activity(model)

    return {
        "model": model,
        "history": pd.DataFrame(history_rows),
        "best_epoch": best_epoch,
        "best_val_ba": best_val_ba,
        "best_val_loss": best_val_loss,
        "firing_onset_epoch": firing_onset_epoch,
        "zero_before": zero_before,
        "zero_after": zero_after,
    }

## 10. Persistent checkpoints / resume

In [ ]:
def objective_tag(objective: str) -> str:
    if objective == "timestep_ce":
        return "timestep"
    if objective == "relative_10bin_ce":
        return "relative10bin"
    raise ValueError(objective)


def checkpoint_path(
    objective: str,
    master_seed: int,
):
    return CHECKPOINT_DIR / (
        f"{objective_tag(objective)}_"
        f"seed_{master_seed}_"
        f"width_{HIDDEN_WIDTH}_best_val_ba.pt"
    )


def history_path(
    objective: str,
    master_seed: int,
):
    return HISTORY_DIR / (
        f"{objective_tag(objective)}_"
        f"seed_{master_seed}_"
        f"width_{HIDDEN_WIDTH}_history.csv"
    )


def checkpoint_contract(
    objective: str,
    master_seed: int,
):
    return {
        "experiment_id": EXPERIMENT_ID,
        "objective": objective,
        "seed": int(master_seed),
        "split_seed": int(SPLIT_SEED),
        "num_epochs": int(NUM_EPOCHS),
        "input_channels": int(INPUT_CHANNELS),
        "num_classes": int(NUM_CLASSES),
        "hidden_width": int(HIDDEN_WIDTH),
        "n_relative_bins": int(N_RELATIVE_BINS),
        "layer1_shifts": list(LAYER1_SHIFTS),
        "layer1_counts": list(LAYER1_COUNTS),
        "layer2_shifts": list(LAYER2_SHIFTS),
        "layer2_counts": list(LAYER2_COUNTS),
        "tau_mem_ms": float(TAU_MEM_MS),
        "tau_syn_out_ms": float(TAU_SYN_OUT_MS),
        "threshold": float(THRESHOLD),
        "sampling_rate_hz": float(SAMPLING_RATE_HZ),
        "encoder_spec_sha256": ENCODER_SPEC_SHA256,
    }


def save_completed_run(
    objective: str,
    master_seed: int,
    run,
):
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    HISTORY_DIR.mkdir(parents=True, exist_ok=True)

    payload = checkpoint_contract(objective, master_seed)
    payload.update(
        {
            "best_epoch": int(run["best_epoch"]),
            "best_val_ba": float(run["best_val_ba"]),
            "best_val_loss": float(run["best_val_loss"]),
            "firing_onset_epoch": run["firing_onset_epoch"],
            "zero_before": run["zero_before"],
            "zero_after": run["zero_after"],
            "state_dict": _cpu_state_dict(run["model"]),
        }
    )

    torch.save(
        payload,
        checkpoint_path(objective, master_seed),
    )

    run["history"].to_csv(
        history_path(objective, master_seed),
        index=False,
    )


def load_completed_run(
    objective: str,
    master_seed: int,
):
    ckpt = checkpoint_path(objective, master_seed)
    hist = history_path(objective, master_seed)

    if not (ckpt.exists() and hist.exists()):
        return None

    payload = torch.load(
        ckpt,
        map_location="cpu",
        weights_only=False,
    )

    required = checkpoint_contract(objective, master_seed)

    for key, expected in required.items():
        if payload.get(key) != expected:
            raise ValueError(
                f"Checkpoint mismatch for {ckpt}: "
                f"{key}={payload.get(key)!r}, expected {expected!r}. "
                "Delete the stale checkpoint or change RESULTS_DIR."
            )

    model = PhaseCMultiTauSNN(
        objective=objective,
        num_classes=NUM_CLASSES,
        input_channels=INPUT_CHANNELS,
        master_seed=master_seed,
    ).to(DEVICE)

    model.load_state_dict(payload["state_dict"])

    return {
        "model": model,
        "history": pd.read_csv(hist),
        "best_epoch": int(payload["best_epoch"]),
        "best_val_ba": float(payload["best_val_ba"]),
        "best_val_loss": float(payload["best_val_loss"]),
        "firing_onset_epoch": payload.get("firing_onset_epoch"),
        "zero_before": payload.get("zero_before", {}),
        "zero_after": payload.get("zero_after", {}),
    }

## 11. Objective $\times$ seed sweep

This cell performs all 10 full trainings. The split is fixed by `SPLIT_SEED`. Within each master
seed, the two objectives use the same DataLoader randomization and the same backbone
initialization.

In [ ]:
run_rows = []
history_frames = []

for objective in OBJECTIVES:
    print("=" * 100)
    print("OBJECTIVE:", objective)
    print("=" * 100)

    for master_seed in SEEDS:
        print("-" * 100)
        print("MASTER SEED:", master_seed)

        loaders = make_split_loaders(
            master_seed,
            include_test=True,
        )

        run = (
            load_completed_run(objective, master_seed)
            if RESUME_EXISTING
            else None
        )

        if run is None:
            run = train_one_configuration(
                objective=objective,
                master_seed=master_seed,
                train_loader=loaders["train"],
                train_eval_loader=loaders["train_eval"],
                val_loader=loaders["val"],
            )
            save_completed_run(
                objective,
                master_seed,
                run,
            )
        else:
            print(
                "Loaded completed checkpoint:",
                checkpoint_path(objective, master_seed),
            )

        model = run["model"]

        train_metrics = evaluate_model(
            model,
            loaders["train_eval"],
        )
        val_metrics = evaluate_model(
            model,
            loaders["val"],
        )

        # Test is touched only after the best epoch was selected on validation BA.
        test_metrics = evaluate_model(
            model,
            loaders["test"],
        )

        row = {
            "experiment": EXPERIMENT_ID,
            "objective": objective,
            "seed": master_seed,
            "best_epoch_by_val_ba": run["best_epoch"],
            "best_val_ba": run["best_val_ba"],
            "val_loss_at_best_val_ba": run["best_val_loss"],
            "firing_onset_epoch": run["firing_onset_epoch"],
            "backbone_parameters": model.backbone_parameter_count,
            "readout_parameters": model.readout_parameter_count,
            "total_parameters": model.total_parameter_count,
        }

        for split_name, metrics in (
            ("train", train_metrics),
            ("val", val_metrics),
            ("test", test_metrics),
        ):
            for key, value in metrics.items():
                row[f"{split_name}_{key}"] = value

        for key, value in run["zero_before"].items():
            row[f"before_{key}"] = value

        for key, value in run["zero_after"].items():
            row[f"after_{key}"] = value

        run_rows.append(row)

        h = run["history"].copy()
        h["objective"] = objective
        h["seed"] = master_seed
        history_frames.append(h)

        print(
            "BEST:",
            f"epoch={run['best_epoch']}",
            f"val BA={val_metrics['balanced_accuracy']:.4f}",
            f"test BA={test_metrics['balanced_accuracy']:.4f}",
            f"test macro-F1={test_metrics['macro_f1']:.4f}",
        )

        del model
        del run

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results_df = pd.DataFrame(run_rows)
histories_df = pd.concat(
    history_frames,
    ignore_index=True,
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    RESULTS_DIR / "experiment_3_0_results.csv",
    index=False,
)
histories_df.to_csv(
    RESULTS_DIR / "experiment_3_0_all_histories.csv",
    index=False,
)

display(results_df)

## 12. Aggregate results across the five paired seeds

In [ ]:
summary_df = (
    results_df
    .groupby("objective")
    .agg(
        seeds=("seed", "nunique"),
        mean_best_val_ba=("best_val_ba", "mean"),
        sd_best_val_ba=("best_val_ba", "std"),
        mean_test_ba=("test_balanced_accuracy", "mean"),
        sd_test_ba=("test_balanced_accuracy", "std"),
        mean_test_macro_f1=("test_macro_f1", "mean"),
        sd_test_macro_f1=("test_macro_f1", "std"),
        mean_test_output_fr=("test_output_firing_rate", "mean"),
        mean_test_l1_fr=("test_hidden_1_firing_rate", "mean"),
        mean_test_l2_fr=("test_hidden_2_firing_rate", "mean"),
    )
    .reset_index()
)

display(summary_df)

summary_df.to_csv(
    RESULTS_DIR / "experiment_3_0_objective_summary.csv",
    index=False,
)

provenance = {
    "experiment_id": EXPERIMENT_ID,
    "split_seed": SPLIT_SEED,
    "seeds": list(SEEDS),
    "objectives": list(OBJECTIVES),
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "hidden_width": HIDDEN_WIDTH,
    "layer1_shifts": list(LAYER1_SHIFTS),
    "layer1_counts": list(LAYER1_COUNTS),
    "layer2_shifts": list(LAYER2_SHIFTS),
    "layer2_counts": list(LAYER2_COUNTS),
    "tau_mem_ms": TAU_MEM_MS,
    "tau_syn_out_ms": TAU_SYN_OUT_MS,
    "threshold": THRESHOLD,
    "sampling_rate_hz": SAMPLING_RATE_HZ,
    "input_channels": INPUT_CHANNELS,
    "num_classes": NUM_CLASSES,
    "event_representation": EVENT_REPRESENTATION,
    "event_feature_schema": EVENT_FEATURE_SCHEMA,
    "encoder_spec_sha256": ENCODER_SPEC_SHA256,
}

with open(
    RESULTS_DIR / "experiment_3_0_provenance.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        provenance,
        f,
        indent=2,
        sort_keys=True,
    )

## 13. Validation balanced accuracy vs. epoch

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

for objective in OBJECTIVES:
    objective_hist = histories_df[
        histories_df.objective == objective
    ]

    # Individual seed trajectories.
    for seed in SEEDS:
        seed_hist = objective_hist[
            objective_hist.seed == seed
        ].sort_values("epoch")

        ax.plot(
            seed_hist["epoch"],
            seed_hist["val_balanced_accuracy"],
            alpha=0.18,
            linewidth=1.0,
        )

    # Mean +/- one SD across the five paired seeds.
    epoch_summary = (
        objective_hist
        .groupby("epoch")["val_balanced_accuracy"]
        .agg(["mean", "std"])
        .reset_index()
    )

    mean_line, = ax.plot(
        epoch_summary["epoch"],
        epoch_summary["mean"],
        linewidth=2.5,
        label=objective,
    )

    ax.fill_between(
        epoch_summary["epoch"],
        (
            epoch_summary["mean"] - epoch_summary["std"]
        ).clip(lower=0.0),
        (
            epoch_summary["mean"] + epoch_summary["std"]
        ).clip(upper=1.0),
        alpha=0.18,
        color=mean_line.get_color(),
    )

ax.axhline(
    1.0 / NUM_CLASSES,
    linestyle="--",
    linewidth=1.0,
    alpha=0.6,
    label="chance",
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Validation balanced accuracy")
ax.set_title("Experiment 3.0: validation BA vs epoch")
ax.set_ylim(
    0.0,
    max(
        0.55,
        histories_df["val_balanced_accuracy"].max() * 1.08,
    ),
)
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()
plt.show()

## 14. Validation/test objective comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))

x = np.arange(len(summary_df))

ax.errorbar(
    x - 0.08,
    summary_df["mean_best_val_ba"],
    yerr=summary_df["sd_best_val_ba"],
    marker="o",
    capsize=4,
    linestyle="none",
    label="validation",
)

ax.errorbar(
    x + 0.08,
    summary_df["mean_test_ba"],
    yerr=summary_df["sd_test_ba"],
    marker="o",
    capsize=4,
    linestyle="none",
    label="test",
)

ax.axhline(
    1.0 / NUM_CLASSES,
    linestyle="--",
    linewidth=1.0,
    alpha=0.6,
    label="chance",
)

ax.set_xticks(
    x,
    summary_df["objective"],
)
ax.set_ylabel("Balanced accuracy")
ax.set_title("Experiment 3.0 objective comparison")
ax.grid(axis="y", alpha=0.25)
ax.legend()

plt.tight_layout()
plt.show()

## 15. Spike raster visualization

For each objective this section selects the seed with the highest validation BA within that
objective, loads its best checkpoint, then visualizes one deterministic validation gesture.

- x-axis: timestep;
- y-axis: individual neuron index;
- each mark: that neuron fired at that timestep;
- horizontal boundaries: changes in hidden-layer `shift_syn` group.

In [ ]:
def load_best_checkpoint_model(
    objective: str,
    master_seed: int,
):
    loaded = load_completed_run(
        objective,
        master_seed,
    )
    if loaded is None:
        raise FileNotFoundError(
            checkpoint_path(
                objective,
                master_seed,
            )
        )
    model = loaded["model"]
    model.eval()
    return model


def get_raster_sample(
    split_name="val",
    index=0,
):
    subset = FIXED_SPLIT_MANIFEST[
        FIXED_SPLIT_MANIFEST.split == split_name
    ]
    dataset = EventSNNDataset(data, subset)
    return dataset[index]


@torch.no_grad()
def run_single_sample(
    model,
    sample,
):
    x = sample["x"][None, ...].to(DEVICE)
    mask = sample["valid_mask"][None, ...].to(DEVICE)

    out = model(x, mask)

    valid_length = int(
        sample["valid_length"].item()
    )

    return {
        "hidden_1": (
            out["hidden_spikes"][0][0, :valid_length]
            .detach()
            .cpu()
            .numpy()
        ),
        "hidden_2": (
            out["hidden_spikes"][1][0, :valid_length]
            .detach()
            .cpu()
            .numpy()
        ),
        "output": (
            out["output_spikes"][0, :valid_length]
            .detach()
            .cpu()
            .numpy()
        ),
        "valid_length": valid_length,
    }


def add_group_boundaries(
    ax,
    groups,
):
    for _, _, stop in groups[:-1]:
        ax.axhline(
            stop - 0.5,
            linestyle="--",
            linewidth=0.8,
            alpha=0.55,
        )


def raster_points(ax, spikes):
    time_idx, neuron_idx = np.nonzero(spikes > 0)

    ax.scatter(
        time_idx,
        neuron_idx,
        s=10,
        marker="|",
    )


def plot_spike_raster(
    model,
    sample,
    *,
    objective: str,
    master_seed: int,
):
    activity = run_single_sample(model, sample)
    valid_length = activity["valid_length"]

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(12, 9),
        sharex=True,
        gridspec_kw={
            "height_ratios": [1, 1, 0.45]
        },
    )

    raster_points(
        axes[0],
        activity["hidden_1"],
    )
    add_group_boundaries(
        axes[0],
        LAYER1_GROUP_SLICES,
    )
    axes[0].set_ylim(-1, HIDDEN_WIDTH)
    axes[0].set_ylabel("L1 neuron")
    axes[0].set_title(
        "Layer 1 spikes — s=2: neurons 0–63; s=3: neurons 64–127"
    )

    raster_points(
        axes[1],
        activity["hidden_2"],
    )
    add_group_boundaries(
        axes[1],
        LAYER2_GROUP_SLICES,
    )
    axes[1].set_ylim(-1, HIDDEN_WIDTH)
    axes[1].set_ylabel("L2 neuron")
    axes[1].set_title(
        "Layer 2 spikes — groups ordered s=2,3,4,5,6,7"
    )

    raster_points(
        axes[2],
        activity["output"],
    )
    axes[2].set_ylim(-1, NUM_CLASSES)
    axes[2].set_yticks(range(NUM_CLASSES))
    axes[2].set_yticklabels(
        [
            IDX_TO_CLASS[i]
            for i in range(NUM_CLASSES)
        ]
    )
    axes[2].set_ylabel("Output class")
    axes[2].set_xlabel("Timestep")
    axes[2].set_title("Output spikes")

    for ax in axes:
        ax.set_xlim(-1, valid_length)
        ax.grid(axis="x", alpha=0.15)

    label_idx = int(sample["label"].item())
    sample_id = sample["sample_id"]

    fig.suptitle(
        f"{objective} | seed={master_seed} | "
        f"true label={IDX_TO_CLASS[label_idx]} | "
        f"{sample_id}",
        y=1.01,
    )

    plt.tight_layout()
    plt.show()


RASTER_SPLIT = "val"
RASTER_SAMPLE_INDEX = 0

raster_sample = get_raster_sample(
    RASTER_SPLIT,
    RASTER_SAMPLE_INDEX,
)

for objective in OBJECTIVES:
    best_row = (
        results_df[
            results_df.objective == objective
        ]
        .sort_values(
            [
                "best_val_ba",
                "val_loss_at_best_val_ba",
            ],
            ascending=[False, True],
        )
        .iloc[0]
    )

    best_seed = int(best_row["seed"])

    print(
        f"{objective}: raster uses seed={best_seed}, "
        f"best val BA={best_row['best_val_ba']:.4f}"
    )

    raster_model = load_best_checkpoint_model(
        objective,
        best_seed,
    )

    plot_spike_raster(
        raster_model,
        raster_sample,
        objective=objective,
        master_seed=best_seed,
    )

    del raster_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 16. Per-$\tau_{\mathrm{syn}}$ firing-rate summary

This checks whether all nominal temporal scales are actually active. A slow group can carry
temporal state while firing less frequently than a fast group, so interpret these rates together
with the raster plots and classification results.

In [ ]:
group_metric_rows = []

for _, row in results_df.iterrows():
    for layer_idx, shifts in (
        (1, LAYER1_SHIFTS),
        (2, LAYER2_SHIFTS),
    ):
        for shift in shifts:
            metric = (
                f"test_hidden_{layer_idx}_"
                f"shift_{shift}_firing_rate"
            )

            group_metric_rows.append(
                {
                    "objective": row["objective"],
                    "seed": int(row["seed"]),
                    "layer": f"L{layer_idx}",
                    "shift": int(shift),
                    "tau_syn_ms": alpha_to_tau_ms(
                        shift_to_alpha(shift),
                        SAMPLING_RATE_HZ,
                    ),
                    "test_firing_rate": row[metric],
                }
            )

group_fr_df = pd.DataFrame(group_metric_rows)

group_fr_summary = (
    group_fr_df
    .groupby(
        [
            "objective",
            "layer",
            "shift",
            "tau_syn_ms",
        ]
    )["test_firing_rate"]
    .agg(["mean", "std"])
    .reset_index()
)

display(group_fr_summary)

group_fr_summary.to_csv(
    RESULTS_DIR / "experiment_3_0_tau_group_firing_rates.csv",
    index=False,
)

for objective in OBJECTIVES:
    fig, ax = plt.subplots(figsize=(8.5, 5))

    sdf = group_fr_summary[
        group_fr_summary.objective == objective
    ]

    for layer_name in ("L1", "L2"):
        ldf = sdf[
            sdf.layer == layer_name
        ].sort_values("tau_syn_ms")

        ax.errorbar(
            ldf["tau_syn_ms"],
            ldf["mean"],
            yerr=ldf["std"],
            marker="o",
            capsize=3,
            label=layer_name,
        )

    ax.set_xscale("log")
    ax.set_xlabel(r"Nominal $\tau_{\mathrm{syn}}$ (ms)")
    ax.set_ylabel("Test firing rate")
    ax.set_title(
        f"{objective}: firing rate by "
        r"$\tau_{\mathrm{syn}}$ group"
    )
    ax.grid(alpha=0.25)
    ax.legend()

    plt.tight_layout()
    plt.show()

## Interpretation checklist

The primary comparison is the paired five-seed difference between `timestep_ce` and
`relative_10bin_ce` under the same multi-$\tau_{\mathrm{syn}}$ backbone.

Check:

- Does validation BA rise consistently above chance for both objectives?
- Which objective has higher mean validation/test BA and lower seed variance?
- Do L1 and L2 maintain healthy firing rates without zero-input firing?
- Are the L2 \(s=6\) and \(s=7\) groups active?
- Do the raster plots show different temporal activity patterns across ordered
  $\tau_{\mathrm{syn}}$ groups?
- Does `timestep_ce` produce more sustained timestep-wise class evidence?
- Does `relative_10bin_ce` benefit more from gesture-phase-specific activity?

A later matched control should compare this 128→128 multi-$\tau$ architecture with a
128→128 single-$\tau$ architecture so any performance change can be attributed to temporal
heterogeneity rather than width.